# Semana 42 · ¿Qué pasó al principio?

Este cuaderno acompaña a la sesión de proyecto de la semana 42: la campana de Gauss y el teorema central del límite, el espectro del fondo cósmico de microondas y un mapa simulado de sus manchas.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. El teorema central del límite con dados

Tiramos 100 000 veces un dado, dos dados y diez dados, y dibujamos la distribución de la suma. Encima, la campana de Gauss con la misma media y la misma desviación típica.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

def gauss(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, n_dados in zip(axs, [1, 2, 10]):
    sumas = rng.integers(1, 7, size=(100_000, n_dados)).sum(axis=1)
    valores = np.arange(n_dados, 6 * n_dados + 2)
    ax.hist(sumas, bins=valores - 0.5, density=True, color="tab:blue", alpha=0.7)
    x = np.linspace(n_dados - 1, 6 * n_dados + 1, 300)
    ax.plot(x, gauss(x, sumas.mean(), sumas.std()), "r")
    ax.set_title(f"suma de {n_dados} dado{'s' if n_dados > 1 else ''}")
plt.tight_layout(); plt.show()

## 2. La regla del 68-95-99,7

In [ ]:
datos = rng.normal(0, 1, 1_000_000)
for k in [1, 2, 3, 5]:
    dentro = (np.abs(datos) < k).mean()
    print(f"dentro de ±{k}σ: {100 * dentro:.4f} %   fuera: {100 * (1 - dentro):.5f} %")

Con un millón de datos, fuera de ±5σ no suele caer ninguno, o apenas uno: la probabilidad es de unas 6 en diez millones.

## 3. El espectro del fondo cósmico

El satélite COBE midió el fondo cósmico en frecuencia, no en longitud de onda, así que usamos la curva de Planck en frecuencia:

$$B_\nu(T) = \frac{2h\nu^3}{c^2}\,\frac{1}{e^{h\nu/(k_B T)} - 1}.$$

Generamos 40 puntos **simulados** entre 60 y 600 GHz con errores parecidos a los de COBE (una parte en diez mil del máximo), y ajustamos la temperatura buscando la que hace mínimo el $\chi^2$ (semana 41).

In [ ]:
h, c, kB = 6.62607015e-34, 2.99792458e8, 1.380649e-23
T_real = 2.7255

def B_nu(nu_ghz, T):
    nu = nu_ghz * 1e9
    return 2 * h * nu**3 / c**2 / np.expm1(h * nu / (kB * T))

nu = np.linspace(60, 600, 40)
error = 1e-4 * B_nu(160, T_real)
datos = B_nu(nu, T_real) + rng.normal(0, error, nu.size)

Ts = np.linspace(2.70, 2.75, 5001)
chi2 = np.array([(((datos - B_nu(nu, T)) / error) ** 2).sum() for T in Ts])
T_mejor = Ts[np.argmin(chi2)]
# incertidumbre: donde chi² sube 1 respecto a su mínimo
dentro = Ts[chi2 < chi2.min() + 1]
print(f"T ajustada = {T_mejor:.5f} K  (+{dentro.max() - T_mejor:.5f} / -{T_mejor - dentro.min():.5f})   real: {T_real}")
print(f"chi² mínimo = {chi2.min():.1f} con {nu.size - 1} grados de libertad")

nu_fino = np.linspace(10, 700, 500)
fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
a1.plot(nu_fino, B_nu(nu_fino, T_mejor) * 1e18, "r", label=f"cuerpo negro a {T_mejor:.4f} K")
a1.errorbar(nu, datos * 1e18, yerr=error * 1e18, fmt="o", ms=3, label="datos simulados")
a1.set_ylabel("B_ν (×10⁻¹⁸ W m⁻² Hz⁻¹ sr⁻¹)"); a1.legend(); a1.grid(alpha=0.3)
a2.errorbar(nu, (datos - B_nu(nu, T_mejor)) / error, yerr=1, fmt="o", ms=3)
a2.axhline(0, color="gray"); a2.set_xlabel("frecuencia (GHz)"); a2.set_ylabel("residuo / error"); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"máximo de la curva: {nu_fino[np.argmax(B_nu(nu_fino, T_real))]:.0f} GHz (λ = {c / 160.2e9 * 1000:.2f} mm)")

El máximo en frecuencia, unos 160 GHz, corresponde a 1,87 mm, no a los 1,06 mm del máximo en longitud de onda de la ley de Wien: es el efecto que viste en el apartado opcional de la semana 31. Con errores tan pequeños, las barras de error ni se ven en la gráfica de arriba: hay que mirar los residuos, divididos por su error, en la de abajo. En el COBE real, los errores eran todavía más pequeños en buena parte del espectro, y los residuos no mostraron ninguna desviación del cuerpo negro.

## 4. La temperatura en el pasado

$$T(z) = T_0\,(1 + z).$$

In [ ]:
for nombre, z in [("hoy", 0), ("galaxias con z = 3", 3), ("galaxias más lejanas del Webb", 14), ("recombinación", 1100)]:
    T = T_real * (1 + z)
    lam_max = 2.898e-3 / T
    texto = f"{lam_max * 1e3:.2f} mm" if lam_max > 1e-3 else (f"{lam_max * 1e6:.1f} µm" if lam_max > 1e-6 else f"{lam_max * 1e9:.0f} nm")
    print(f"{nombre:<32} z = {z:5d}   T = {T:8.1f} K   máximo de Wien en {texto}")

En la recombinación, el máximo estaba en el infrarrojo cercano, cerca de 1 µm, pero la curva cubría también buena parte del visible: el universo entero brillaba como el filamento de una bombilla, con un tono anaranjado.

## 5. Un mapa simulado de las manchas

Generamos un trozo de cielo de 20° × 20° con manchas aleatorias gaussianas de un tamaño típico de alrededor de un grado y una desviación típica de 50 µK. Para hacerlo, partimos de **ruido blanco** (un valor al azar independiente en cada píxel) y lo suavizamos con un filtro gaussiano usando la transformada de Fourier (semana 27): es la forma de imponer un tamaño típico a las manchas. **Es una simulación sencilla**; los mapas reales tienen un espectro de potencias con varios picos.

In [ ]:
n_pix, tam_grados = 400, 20.0
ruido = rng.normal(0, 1, (n_pix, n_pix))
k = np.fft.fftfreq(n_pix, d=tam_grados / n_pix)           # ciclos por grado
kx, ky = np.meshgrid(k, k)
filtro = np.exp(-0.5 * (np.hypot(kx, ky) * 2 * np.pi * 0.35) ** 2)   # suavizado de unos 0,35° de sigma
mapa = np.real(np.fft.ifft2(np.fft.fft2(ruido) * filtro))
mapa = mapa / mapa.std() * 50e-6                          # desviación típica de 50 µK

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5.5))
im = a1.imshow(mapa * 1e6, cmap="RdBu_r", extent=(0, tam_grados, 0, tam_grados), vmin=-150, vmax=150)
plt.colorbar(im, ax=a1, label="ΔT (µK)"); a1.set_xlabel("grados"); a1.set_title("Fondo cósmico simulado (20° × 20°)")
a2.hist(mapa.ravel() * 1e6, bins=80, density=True, color="tab:purple", alpha=0.7)
x = np.linspace(-200, 200, 300)
a2.plot(x, gauss(x, 0, 50), "k"); a2.set_xlabel("ΔT (µK)"); a2.set_title("Histograma de las temperaturas")
plt.tight_layout(); plt.show()
print(f"desviación típica: {mapa.std() * 1e6:.1f} µK;  fuera de ±3σ: {100 * (np.abs(mapa) > 3 * mapa.std()).mean():.2f} %")

## 6. El dipolo de nuestro movimiento

El Sol se mueve a unos 370 km/s respecto al fondo cósmico, hacia un punto de longitud galáctica 264° y latitud 48°. Por el efecto Doppler, el fondo se ve un poco más caliente en esa dirección y más frío en la opuesta: $\Delta T = T_0\,\frac{v}{c}\cos\theta$, donde $\theta$ es el ángulo con la dirección del movimiento.

In [ ]:
v = 370e3
l = np.radians(np.linspace(-180, 180, 361))
b = np.radians(np.linspace(-90, 90, 181))
L, B = np.meshgrid(l, b)
l0, b0 = np.radians(264 - 360), np.radians(48)
cos_theta = np.sin(B) * np.sin(b0) + np.cos(B) * np.cos(b0) * np.cos(L - l0)
dT = T_real * v / c * cos_theta

fig = plt.figure(figsize=(11, 6))
ax = fig.add_subplot(projection="mollweide")
im = ax.pcolormesh(-L, B, dT * 1e3, cmap="RdBu_r", shading="auto")
plt.colorbar(im, orientation="horizontal", pad=0.06, label="ΔT (mK)")
ax.set_xticklabels([]); ax.set_title("El dipolo del fondo cósmico (coordenadas galácticas)")
plt.show()
print(f"amplitud del dipolo: {T_real * v / c * 1e3:.2f} mK")

El dipolo es unas cien veces mayor que las manchas: antes de estudiarlas, hay que restarlo. También hay que restar la emisión de la propia Vía Láctea, que es lo que hacen los mapas como el de Planck usando varias frecuencias.

## 7. Para ir más lejos: cuántos fotones hay

Integramos numéricamente la densidad de fotones de un cuerpo negro y la comparamos con la fórmula $n_\gamma = 16\pi\zeta(3)(k_BT/hc)^3$.

In [ ]:
integrar = getattr(np, "trapezoid", None) or np.trapz
x = np.linspace(1e-6, 50, 200_000)
integral = integrar(x**2 / np.expm1(x), x)                  # vale 2·ζ(3)
zeta3 = integral / 2
n_gamma = 8 * np.pi * (kB * T_real / (h * c)) ** 3 * integral   # fotones por m³
print(f"ζ(3) ≈ {zeta3:.5f}")
print(f"fotones del fondo cósmico: {n_gamma / 1e6:.0f} por cm³")

## 8. Experimenta

1. Suma 10 valores de una distribución muy asimétrica (por ejemplo, `rng.exponential(1, ...)`). ¿La suma se parece a una campana? ¿Y con 50?
2. En el ajuste del espectro, multiplica los errores por 100. ¿Cuánto empeora la precisión de la temperatura?
3. En el mapa simulado, cambia el suavizado a 0,1° y a 1°. ¿Cómo cambian las manchas? ¿Y el histograma?

In [ ]:
# Tu experimento aquí